# Forensic Replication — Shadow Hunter & Manipulation Analysis

> **Requires data**: ThetaData V3 API (options tick data) + Polygon.io (equity data).  
> This notebook re-runs the forensic analysis scripts and compares live results to pre-computed JSONs.

## Prerequisites

1. **Polygon API key**: `export POLYGON_API_KEY=<your_key>`
2. **ThetaData Theta Terminal**: Running on `localhost:25510`
3. **Data**: Pre-fetched to local Parquet (see Data Download section below)

If you don't have API keys, use `00_evidence_viewer.ipynb` instead — it reads pre-computed results.

In [1]:
import json
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# Setup paths
REVIEW_PKG = Path('.').resolve()
CODE_DIR = REVIEW_PKG.parents[0] / 'code'
RESULTS_DIR = REVIEW_PKG.parents[0] / 'results'
sys.path.insert(0, str(CODE_DIR))

# Verify structure
assert CODE_DIR.exists(), f'code/ not found at {CODE_DIR}'
assert RESULTS_DIR.exists(), f'results/ not found at {RESULTS_DIR}'
print(f'Scripts: {len(list(CODE_DIR.glob("*.py")))} Python files')
print(f'Results: {len(list(RESULTS_DIR.glob("*.json")))} pre-computed JSONs')


def run_streaming(script_path, extra_args=None, timeout_min=10):
    """Run a script with real-time streaming output."""
    cmd = [sys.executable, '-u', str(script_path)]
    if extra_args:
        cmd.extend(extra_args)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, cwd=str(CODE_DIR), env=env
    )
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
        proc.wait(timeout=timeout_min * 60)
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f'\n⚠️  Timed out after {timeout_min} minutes.')
    return proc.returncode

print('\n✅ Helper function run_streaming() defined.')

Scripts: 30 Python files
Results: 113 pre-computed JSONs

✅ Helper function run_streaming() defined.


---
## 1. Environment Check

Verify API keys and data directory structure before running analysis.

In [2]:
# Check Polygon API
polygon_key = os.environ.get('POLYGON_API_KEY')
print(f'Polygon API key: {"✅ Set" if polygon_key else "❌ Not set — equity data fetch will fail"}')

# Check ThetaData
try:
    import socket
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2)
    result = s.connect_ex(('0.0.0.0', 25503))
    s.close()
    theta_ok = result == 0
except Exception:
    theta_ok = False
print(f'ThetaData Terminal: {"✅ Running" if theta_ok else "❌ Not running on 0.0.0.0:25510"}')

# Check data directories
DATA_ROOT = REVIEW_PKG.parents[1] / 'data' / 'raw'
polygon_data = DATA_ROOT / 'polygon' / 'trades'
theta_data = DATA_ROOT / 'thetadata' / 'trades'

if polygon_data.exists():
    eq_tickers = [d.name for d in polygon_data.iterdir() if d.is_dir()]
    print(f'Polygon equity data: ✅ {len(eq_tickers)} tickers')
else:
    print(f'Polygon equity data: ❌ Not found')

if theta_data.exists():
    opt_tickers = [d.name for d in theta_data.iterdir() if d.is_dir()]
    print(f'ThetaData options data: ✅ {len(opt_tickers)} tickers')
else:
    print(f'ThetaData options data: ❌ Not found')

print(f'\nData root: .../{DATA_ROOT.relative_to(DATA_ROOT.parents[2])}')

Polygon API key: ✅ Set
ThetaData Terminal: ✅ Running
Polygon equity data: ✅ 37 tickers
ThetaData options data: ✅ 54 tickers

Data root: .../research/data/raw


---
## 2. Data Download (Equity Only — Polygon)

Auto-fetch GME equity tick data if missing. Options data requires ThetaData Terminal.

In [3]:
# Auto-fetch equity data if Polygon API key is set
if polygon_key and not (polygon_data / 'symbol=GME').exists():
    print('Fetching GME equity data from Polygon...')
    rc = run_streaming(CODE_DIR / 'fetch_early_data.py')
elif not polygon_key:
    print('⚠️  Polygon API key not set — using pre-computed results only.')
    print('    Set POLYGON_API_KEY to auto-download equity data.')
else:
    gme_days = len(list((polygon_data / 'symbol=GME').glob('date=*/part-0.parquet')))
    print(f'GME equity data already present: {gme_days} trading days')

GME equity data already present: 2037 trading days


---
## 3. Shadow Algorithm Hunter

Runs all 6 smoking gun detectors: wash cross, tail banging, COB routing,
algorithmic stepping, dark venue routing.

In [4]:
# Run shadow hunter (requires ThetaData options data present)
gme_opts_dir = theta_data / 'root=GME' if theta_data.exists() else None
has_gme_opts = gme_opts_dir and gme_opts_dir.exists()

if has_gme_opts:
    for event in ['jan2021', 'jun2024']:
        print(f'\n{"="*60}')
        print(f'Running Shadow Hunter -- {event}')
        print(f'{"="*60}')
        rc = run_streaming(
            CODE_DIR / 'shadow_hunter.py',
            ['--ticker', 'GME', '--event', event],
            timeout_min=10
        )
else:
    print('Options data not found -- loading pre-computed results.')
    sh_files = sorted(RESULTS_DIR.glob('shadow_hunter_GME_*.json'))
    if sh_files:
        with open(sh_files[-1]) as f:
            sh_data = json.load(f)
        for event_key, event_data in sh_data.items():
            tests = event_data.get('tests', {})
            print(f'\nPre-computed results -- {event_key} ({sh_files[-1].name}):')
            for test_name, test_data in tests.items():
                print(f'  {test_name}: {test_data.get("verdict", "N/A")}')



Running Shadow Hunter -- jan2021

  TEST G — TAIL-BANGING: GME on 20210127
  Spot ≈ $347.51 | OTM threshold ≥ 50%

  Total call volume:     344,980
  Deep OTM tail trades:  0 trades, 0 lots
  Capital burned:        $0.00
  % of call volume:      0.00%

  VERDICT: MINIMAL TAIL ACTIVITY

  TEST G — TAIL-BANGING: GME on 20210128
  Spot ≈ $193.60 | OTM threshold ≥ 50%

  Total call volume:     213,805
  Deep OTM tail trades:  518 trades, 10,686 lots
  Capital burned:        $69,845,451.00
  % of call volume:      5.00%

  Strike concentration:
    $320.0: 188 trades, 3,423 lots, $31,817,803 burned | Exchanges: MULTI_EXCHANGE, EDGX_OPTIONS, C2, C2_COB, NYSE_AMEX, NSDQ_ISE_GEMINI, BATS, CBOE, BZX_OPTIONS, EDGX_COB, PHLX_FLOOR, ISE, NSDQ_BX_OPT, MIAX_PEARL_EQUITIES
    $570.0: 74 trades, 2,788 lots, $12,340,052 burned | Exchanges: MULTI_EXCHANGE, NYSE_AMEX, PHLX_FLOOR, BATS, CBOE, BZX_OPTIONS, EDGX_COB, NSDQ_BX_OPT, ISE, EDGX_OPTIONS, MIAX_PEARL_EQUITIES, C2_COB
    $300.0: 53 trades, 854 lo

---
## 4. Manipulation Forensic Battery

Runs Tests A–F: whale detector, ignition sequence, constructor fingerprint,
predator matrix, Lee-Ready aggressor, Vanna lag.

In [5]:
# Run manipulation forensic for both events
if has_gme_opts:
    for event in ['jan2021', 'jun2024']:
        print(f'\n{"="*60}')
        print(f'Running Manipulation Forensic — {event}')
        print(f'{"="*60}')
        rc = run_streaming(
            CODE_DIR / 'manipulation_forensic.py',
            ['--event', event, '--ticker', 'GME'],
            timeout_min=10
        )
else:
    print('⚠️  Options data not found — loading pre-computed results.')
    mf_files = sorted(RESULTS_DIR.glob('manipulation_forensic_GME_*.json'))
    for mf_file in mf_files:
        with open(mf_file) as f:
            mf = json.load(f)
        events = mf if isinstance(mf, list) else [mf]
        for ev in events:
            print(f'\n{ev.get("event", "Unknown")} ({mf_file.name}):')
            for tn, td in ev.get('tests', {}).items():
                print(f'  {tn}: {td.get("verdict", "N/A")[:70]}')


Running Manipulation Forensic — jan2021

  WHALE DETECTOR: GME
  Dates: 20210122, 20210125, 20210126, 20210127, 20210128, 20210129
  Target expiration: 20210129

  20210122: 237,208 contracts in 62,479 trades (avg 3.8 lots)
  Whale Ratio (Block+Whale/Total): 14.6%
  Bucket                 Trades     Volume    %Vol  %Trades
  -------------------------------------------------------
  Retail (1-3)           50,181     66,182   27.9%    80.3%
  Small (4-9)             7,570     42,129   17.8%    12.1%
  Medium (10-49)          4,209     71,104   30.0%     6.7%
  Large (50-99)             347     23,050    9.7%     0.6%
  Block (100-499)           158     26,023   11.0%     0.3%
  Whale (500+)               14      8,720    3.7%     0.0%

  Top 5 Largest Trades:
    2021-01-22T14:32:02.809     839 lots  $60  @$10.20  MULTI_EXCHANGE
    2021-01-22T14:45:10.439     800 lots  $8  @$51.65  NYSE_AMEX
    2021-01-22T15:37:32.191     800 lots  $8  @$53.30  NYSE_AMEX
    2021-01-22T14:45:10.439   

---
## 5. Squeeze Mechanics Forensic

Jan 2021 squeeze: wall failures, dealer delta, price magnetism.

In [6]:
if has_gme_opts:
    print('Running Squeeze Mechanics Forensic...')
    print('(5 analyses on Jan 2021 squeeze dates — expect 3-5 min)\n')
    rc = run_streaming(
        CODE_DIR / 'squeeze_mechanics_forensic.py',
        timeout_min=10
    )
else:
    print('⚠️  Loading pre-computed squeeze mechanics results.')
    sm_files = sorted(RESULTS_DIR.glob('squeeze_mechanics_GME_*.json'))
    if sm_files:
        with open(sm_files[-1]) as f:
            sm = json.load(f)
        print(f'Source: {sm_files[-1].name}')
        print(json.dumps(sm, indent=2, default=str)[:2000])

Running Squeeze Mechanics Forensic...
(5 analyses on Jan 2021 squeeze dates — expect 3-5 min)


  ANALYSIS 1: STRIKE LADDER CASCADE
  Scanning 74 trading days (2020-10-30 → 2021-02-17)
    10/74    20/74    30/74    40/74    50/74    60/74    70/74

  STRIKE LADDER BREAKTHROUGHS:
  Date           Strike    Price  Days Wall   Call Vol
  -------------------------------------------------------
  2020-11-02   $     11 $  11.09        3d      2,224
  2020-11-04   $     12 $  12.05        5d      4,460
  2020-11-04   $     12 $  12.05        5d      2,276
  2020-11-09   $     12 $  12.95       10d      3,302
  2020-11-20   $     13 $  13.21       21d     12,263
  2020-11-23   $     14 $  13.67       20d      8,629
  2020-11-24   $     14 $  14.10       25d      9,407
  2020-11-24   $     14 $  14.50       25d      3,243
  2020-11-27   $     15 $  16.31       28d     15,520
  2020-11-27   $     16 $  16.31       28d      8,041
  2020-11-27   $     16 $  17.37       25d     11,855
  2020-11-27

---
## 6. Counterfactual Analysis

Compares squeeze-period metrics vs calm-period baselines.

In [7]:
if has_gme_opts:
    print('Running Counterfactual Analysis...')
    print('(Calm-period baselines vs squeeze — expect 2-3 min)\n')
    rc = run_streaming(
        CODE_DIR / 'counterfactual_analysis.py',
        timeout_min=10
    )
else:
    print('⚠️  Loading pre-computed counterfactual results.')
    cf_files = sorted(RESULTS_DIR.glob('counterfactual_GME_*.json'))
    if cf_files:
        with open(cf_files[-1]) as f:
            cf = json.load(f)
        print(f'Source: {cf_files[-1].name}')
        print(json.dumps(cf, indent=2, default=str)[:2000])

Running Counterfactual Analysis...
(Calm-period baselines vs squeeze — expect 2-3 min)


  ANALYZING: GME Squeeze (Jan 2021)
  Center date: 2021-01-28, Window: -90d to +20d
  Options days: 74, Equity days: 74

  --- Wall Failure Analysis ---
    20/74    40/74    60/74
  Wall tests: 36
  Breached: 23 (63.9%)
  Cascades: 8 (22.2%)

  --- Dealer Delta Range ---
    20/74    40/74    60/74
  Dealer delta range: -30,779,375 to +8,106,011
  Mean: -1,306,829, Std: 5,076,891
  Max |delta|: 30,779,375
  Sign changes: 16

  --- Pin Distance Distribution ---
  Mean pin distance: 21.01%
  Median: 6.82%, Std: 36.31%
  90th percentile: 43.50%

  ANALYZING: GME Calm 2022 Q1
  Center date: 2022-03-15, Window: -90d to +20d
  Options days: 76, Equity days: 76

  --- Wall Failure Analysis ---
    20/76    40/76    60/76
  Wall tests: 31
  Breached: 9 (29.0%)
  Cascades: 6 (19.4%)

  --- Dealer Delta Range ---
    20/76    40/76    60/76
  Dealer delta range: -1,585,173 to +811,383
  Mean: -67,310, Std: 

---
## 7. Rogue Wave — LEAPS Accumulation Ramp

Analyzes temporal build-up of call vs. put open interest before squeeze peaks.
Identifies the critical mass date and explosion ratio.

In [12]:
# Rogue Wave analysis
rw_script = CODE_DIR / 'rogue_wave_detector.py'
if rw_script.exists() and has_gme_opts:
    print('Running Rogue Wave Detector...')
    print('(LEAPS accumulation ramp analysis — expect 2-4 min)\n')
    rc = run_streaming(rw_script, timeout_min=10)
else:
    print('Loading pre-computed rogue wave results.')
    rw_files = sorted(RESULTS_DIR.glob('rogue_wave_GME_*.json'))
    rw = None
    for rwf in rw_files:
        with open(rwf) as f:
            candidate = json.load(f)
        if 'peak_date' in candidate:
            rw = candidate
            print(f'Source: {rwf.name}')
            break
    if rw:
        cm = rw.get('critical_mass', {})
        if cm:
            print(f'  Critical mass date: {cm.get("critical_mass_date", "N/A")}')
            print(f'  Days before peak: {cm.get("days_before_peak", "N/A")}')
            print(f'  Call/put ratio at peak: {cm.get("call_put_ratio_at_peak", "N/A")}')
            print(f'  Final 7d explosion ratio: {cm.get("final_7d_explosion_ratio", "N/A")}')
    elif rw_files:
        print(f'⚠️  Found {len(rw_files)} rogue wave files but none contain peak_date')
    else:
        print('❌ No rogue_wave results found')


Loading pre-computed rogue wave results.
Source: rogue_wave_GME_20260212_094501.json
  Critical mass date: 2020-04-14
  Days before peak: 290
  Call/put ratio at peak: 0.7
  Final 7d explosion ratio: 12.15


---
## 8. Multi-Event Wall Fatigue

Compares wall breach and cascade rates across Jan 2021 and Jun 2024 squeeze events,
testing whether repeated testing of gamma walls causes fatigue.

In [9]:
# Multi-Event Wall Fatigue
wf_script = CODE_DIR / 'multi_event_wall_fatigue.py'
if wf_script.exists() and has_gme_opts:
    print('Running Multi-Event Wall Fatigue...')
    print('(Cross-event wall comparison — expect 2-3 min)\n')
    rc = run_streaming(wf_script, timeout_min=10)
else:
    print('⚠️  Loading pre-computed wall fatigue results.')
    wf_files = sorted(RESULTS_DIR.glob('multi_event_wall_fatigue_*.json'))
    if wf_files:
        with open(wf_files[-1]) as f:
            wf = json.load(f)
        print(f'Source: {wf_files[-1].name}')
        events = wf.get('events', [])
        for ev in events:
            print(f'  {ev.get("ticker","")} {ev.get("peak_date","")}: '
                  f'{ev.get("n_cascade",0)}/{ev.get("n_wall_tests",0)} cascades '
                  f'({ev.get("cascade_rate",0):.1%})')
        comp = wf.get('comparison', {})
        if comp:
            print(f'  Mean cascade rate: {comp.get("mean_cascade_rate", "N/A")}')
    else:
        print('❌ No wall fatigue results found')


Running Multi-Event Wall Fatigue...
(Cross-event wall comparison — expect 2-3 min)


  WALL FATIGUE ANALYSIS: GME (peak: 2021-01-28)
  Equity days: 74
    10/74    20/74    30/74    40/74    50/74    60/74    70/74

  --- Wall Test Results ---

  Date            Close     Wall   Approach       Result     Energy
  ----------------------------------------------------------------
  2020-11-04   $  11.16 $    12     7.55% 🟢 HELD           81,370
  2020-11-05   $  11.63 $    12     3.18% 🟢 HELD           99,177
  2020-11-06   $  11.68 $    12     2.74% 🟢 HELD          312,615
  2020-11-09   $  12.08 $    13     7.58% 🟢 HELD          190,746
  2020-11-12   $  11.71 $    12     2.48% 🟢 HELD          146,674
  2020-11-13   $  11.16 $    12     7.49% 🟢 HELD          170,029
  2020-11-16   $  11.96 $    12     0.29% 🟢 HELD          721,526
  2020-11-17   $  11.32 $    12     6.02% 🟢 HELD          290,238
  2020-11-18   $  11.87 $    12     1.07% 🟢 HELD          157,602
  2020-11-19   $  11.77 $ 

---
## 9. Verification — Live vs Pre-Computed

If you ran live analysis above, this cell compares your results to the
pre-computed JSONs to confirm reproducibility.

In [13]:
# Semantic verification - compare analytical conclusions, not timestamps
import json
from math import isclose

def extract_verdicts(obj, prefix=''):
    verdicts = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'verdict' and isinstance(v, str):
                verdicts[prefix.rstrip('.')] = v
            elif k in ('timestamp', 'runtime_seconds'):
                continue
            elif isinstance(v, (dict, list)):
                verdicts.update(extract_verdicts(v, f'{prefix}{k}.'))
    elif isinstance(obj, list):
        for idx, item in enumerate(obj):
            verdicts.update(extract_verdicts(item, f'{prefix}[{idx}].'))
    return verdicts

def get_nested(obj, key_path):
    parts = key_path.split('.')
    cur = obj
    for p in parts:
        if p.startswith('[') and p.endswith(']'):
            idx = int(p[1:-1])
            cur = cur[idx] if isinstance(cur, list) and idx < len(cur) else None
        elif isinstance(cur, dict):
            cur = cur.get(p)
        else:
            return None
        if cur is None:
            return None
    return cur

def get_event_key(data):
    if isinstance(data, dict):
        keys = [k for k in data if k not in ('analysis','ticker','timestamp','comparison')]
        if keys:
            return keys[0]
    elif isinstance(data, list) and data:
        return data[0].get('event', 'unknown')
    return 'unknown'

def find_matching_pair(files, target_event=None):
    if len(files) < 2:
        return None
    by_event = {}
    for f in files:
        with open(f) as fh:
            data = json.load(fh)
        ek = get_event_key(data)
        by_event.setdefault(ek, []).append((f, data))
    results = []
    for ek, pairs in sorted(by_event.items()):
        if len(pairs) >= 2:
            results.append((ek, pairs[0], pairs[-1]))
    return results

print('Verification — comparing analytical conclusions:')
print()

for prefix in ['shadow_hunter_GME', 'manipulation_forensic_GME',
               'squeeze_mechanics_GME', 'counterfactual_GME',
               'rogue_wave_GME', 'multi_event_wall_fatigue']:
    files = sorted(RESULTS_DIR.glob(f'{prefix}_*.json'))
    if not files:
        print(f'{prefix}: No results found')
        print()
        continue

    if prefix in ('squeeze_mechanics_GME', 'counterfactual_GME',
                   'rogue_wave_GME', 'multi_event_wall_fatigue'):
        if len(files) < 2:
            print(f'{prefix}: ✅ Baseline available ({files[0].name})')
            print()
            continue
        with open(files[0]) as f:
            baseline = json.load(f)
        with open(files[-1]) as f:
            latest = json.load(f)

        bv = extract_verdicts(baseline)
        lv = extract_verdicts(latest)
        v_matches = sum(1 for k in set(bv) & set(lv) if bv[k] == lv[k])
        v_changed = [(k, bv[k], lv[k]) for k in set(bv) & set(lv) if bv[k] != lv[k]]

        n_matches = 0
        n_diffs = []
        checks = []
        if 'counterfactual' in prefix:
            checks = [
                ('periods.[0].wall_breach_rate', 'Squeeze breach rate'),
                ('comparison.breach_z', 'Breach z-score'),
            ]
        elif 'squeeze' in prefix:
            checks = [
                ('implied_delta_exposure.summary.max_abs_dealer_delta', 'Max |Dealer Δ|'),
                ('failed_wall_forensic.summary.total_walls', 'Total walls'),
            ]
        elif 'rogue_wave' in prefix:
            checks = [
                ('critical_mass.days_before_peak', 'Days before peak'),
                ('critical_mass.call_put_ratio_at_peak', 'Call/put ratio'),
            ]
        elif 'wall_fatigue' in prefix:
            checks = [
                ('comparison.mean_cascade_rate', 'Mean cascade rate'),
                ('comparison.n_events', 'N events'),
            ]
        for key_path, display_name in checks:
            b = get_nested(baseline, key_path)
            l = get_nested(latest, key_path)
            if b is None or l is None:
                continue
            if isinstance(b, (int, float)) and isinstance(l, (int, float)):
                if isclose(b, l, rel_tol=0.05, abs_tol=1e-6):
                    n_matches += 1
                else:
                    n_diffs.append((display_name, b, l))
            elif b == l:
                n_matches += 1
            else:
                n_diffs.append((display_name, b, l))

        total_m = v_matches + n_matches
        all_diffs = v_changed + n_diffs
        if not all_diffs:
            print(f'{prefix}: ✅ Reproduced ({total_m}/{total_m} conclusions match)')
        else:
            print(f'{prefix}: ⚠️ {total_m} match, {len(all_diffs)} changed')
            for key, bval, lval in all_diffs[:5]:
                print(f'    {key}: {str(bval)[:40]} -> {str(lval)[:40]}')
        print(f'  Baseline: {files[0].name}')
        print(f'  Latest:   {files[-1].name}')
        print()
    else:
        pairs = find_matching_pair(files)
        if not pairs:
            print(f'{prefix}: Could not find matching event pairs')
            print()
            continue
        for event_key, (bf, bdata), (lf, ldata) in pairs:
            bv = extract_verdicts(bdata)
            lv = extract_verdicts(ldata)
            shared = set(bv) & set(lv)
            matched = sum(1 for k in shared if bv[k] == lv[k])
            changed = [(k, bv[k], lv[k]) for k in shared if bv[k] != lv[k]]
            added = sorted(set(lv) - set(bv))
            removed = sorted(set(bv) - set(lv))

            if not changed and not removed:
                extra = f' (+{len(added)} new tests)' if added else ''
                print(f'{prefix} [{event_key}]: ✅ Reproduced '
                      f'({matched}/{matched} verdicts match{extra})')
            else:
                print(f'{prefix} [{event_key}]: ⚠️ {matched} match, '
                      f'{len(changed)} changed, {len(removed)} removed')
                for key, bval, lval in changed[:3]:
                    print(f'    Changed: {key.split(".")[-1]}: '
                          f'{str(bval)[:35]} -> {str(lval)[:35]}')
            print(f'  Baseline: {bf.name}')
            print(f'  Latest:   {lf.name}')
            print()

print('Done. Verdicts and key metrics compared; timestamps ignored.')


Verification — comparing analytical conclusions:

shadow_hunter_GME [jan_2021]: ✅ Reproduced (16/16 verdicts match)
  Baseline: shadow_hunter_GME_20260212_145054.json
  Latest:   shadow_hunter_GME_20260216_122851.json

shadow_hunter_GME [jun_2024]: ✅ Reproduced (21/21 verdicts match)
  Baseline: shadow_hunter_GME_20260212_145136.json
  Latest:   shadow_hunter_GME_20260216_122900.json

manipulation_forensic_GME [Jan 2021 Squeeze]: ✅ Reproduced (3/3 verdicts match (+3 new tests))
  Baseline: manipulation_forensic_GME_20260212_131156.json
  Latest:   manipulation_forensic_GME_20260216_122907.json

manipulation_forensic_GME [Jun 2024 Squeeze]: ✅ Reproduced (3/3 verdicts match (+3 new tests))
  Baseline: manipulation_forensic_GME_20260212_131218.json
  Latest:   manipulation_forensic_GME_20260216_122910.json

squeeze_mechanics_GME: ✅ Reproduced (0/0 conclusions match)
  Baseline: squeeze_mechanics_GME_20260212_104815.json
  Latest:   squeeze_mechanics_GME_20260216_123338.json

counterfactua

---

**Done.** If any tests were skipped due to missing data, install the relevant APIs
and re-run, or review the pre-computed results in `00_evidence_viewer.ipynb`.
